# Coordinated data augmentation

Use the real `mprage_hippmapp3r` T1 from the augmentation section of the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621).
The image downloads on first use. A central axial slice is resized to 96 × 96
for this short illustration; no artificial anatomical phantom is generated.
ANTsTorch delegates augmentation to ANTsPy on CPU.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
t1 = ants.image_read(antstorch.get_antstorch_data("mprage_hippmapp3r")).clone("float")
reference = ants.slice_image(t1, axis=2, idx=t1.shape[2] // 2)
reference = ants.resample_image(reference, (96, 96), use_voxels=True)
reference = ants.iMath(reference, "Normalize")
segmentation = ants.get_mask(reference)  # Estimated foreground mask, not manual labels.
# Duplicate modalities make shared spatial sampling easy to check.
modalities = [reference.clone(), reference.clone()]
with time_step('Data augmentation'):
    augmented, augmented_labels = antstorch.augment_image_modalities(
        modalities, reference_image=reference, transform_type="affine",
        sd_affine=0.03, sd_deformation=0,
        noise_model=None, sd_simulated_bias_field=0, sd_histogram_warping=0,
        segmentation_image=[segmentation],
    )
assert len(augmented) == 2 and augmented_labels is not None
np.testing.assert_allclose(augmented[0].numpy(), augmented[1].numpy(), atol=1e-6)
assert np.isin(augmented_labels.numpy(), [0, 1]).all()
for image in [*augmented, augmented_labels]:
    assert image.shape == reference.shape and np.isfinite(image.numpy()).all()
    np.testing.assert_allclose(image.spacing, reference.spacing)
    np.testing.assert_allclose(image.origin, reference.origin)
    np.testing.assert_allclose(image.direction, reference.direction)

## Inspect the transformed label boundary
The segmentation uses label interpolation, while intensities use image
interpolation. Intensity normalization still occurs inside augmentation even
with noise, bias and histogram perturbations disabled. Seeds help reproduce
Python/NumPy choices; native ANTs randomness need not be identical across runs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, image, mask, title in zip(
    axes, [reference, *augmented], [segmentation, augmented_labels, augmented_labels],
    ["Original", "Augmented modality A", "Augmented modality B"],
):
    ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.contour(mask.numpy().T, levels=[0.5], colors="red", linewidths=1)
    ax.set_title(title); ax.axis("off")
plt.show()

## Add intensity variation
Spatial augmentation should stay synchronized across modalities and labels.
Intensity perturbations can differ between channels; they must not modify
the label identities. Here noise and histogram warping are enabled without
adding a spatial deformation.

In [ ]:
with time_step('Data augmentation'):
    perturbed, _ = antstorch.augment_image_modalities(
        modalities, reference_image=reference, transform_type="affine", sd_affine=0,
        noise_model="additivegaussian", noise_parameters=(0., 0.03),
        sd_simulated_bias_field=0, sd_histogram_warping=0.02,
    )
assert all(np.isfinite(image.numpy()).all() for image in perturbed)
fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)
for ax, image, title in zip(axes, perturbed, ["Intensity variant A", "Intensity variant B"]):
    ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()

The duplicated channels contain the same acquired MRI slice and make shared
spatial sampling easy to check; they are not two independently acquired
modalities. For a multimodal dataset, pass genuinely aligned acquisitions and
their matching labels. The displayed foreground mask is estimated from this
image and is not ground truth. Choose plausible augmentation ranges for training.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")